<a href="https://colab.research.google.com/github/ML-HW-SYS/a3-AaronGoldblatt/blob/main/5-conv2d_dw_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Depthwise-seperable 2D Convolution on GPU

## 1. Set-up

In [1]:
# Mount google drive
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [2]:
# Make sure your token is stored in a txt file at the location below.
# This way there is no risk that you will push it to your repo
# Never share your token with anyone, it is basically your github password!
with open('/content/gdrive/MyDrive/ece5545/token.txt') as f:
    token = f.readline().strip()
# Use another file to store your github username
with open('/content/gdrive/MyDrive/ece5545/git_username.txt') as f:
    handle = f.readline().strip()

In [10]:
# Clone your github repo
YOUR_TOKEN = token
YOUR_HANDLE = handle
BRANCH = "main"

%mkdir /content/gdrive/MyDrive/ece5545
%cd /content/gdrive/MyDrive/ece5545
!git clone https://{YOUR_TOKEN}@github.com/ML-HW-SYS/a3-{YOUR_HANDLE}.git
%cd /content/gdrive/MyDrive/ece5545/a3-{YOUR_HANDLE}
!git checkout {BRANCH}
!git pull
%cd /content/gdrive/MyDrive/ece5545

PROJECT_ROOT = f"/content/gdrive/MyDrive/ece5545/a3-{YOUR_HANDLE}"

mkdir: cannot create directory ‘/content/gdrive/MyDrive/ece5545’: File exists
/content/gdrive/MyDrive/ece5545
fatal: destination path 'a3-AaronGoldblatt' already exists and is not an empty directory.
/content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
Already on 'main'
Your branch is up to date with 'origin/main'.
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 8 (delta 4), reused 7 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.05 KiB | 21.00 KiB/s, done.
From https://github.com/ML-HW-SYS/a3-AaronGoldblatt
   a95f561..4463264  main       -> origin/main
Updating a95f561..4463264
Fast-forward
 src/ops.py | 31 ++++++++++++++++++++++---------
 1 file changed, 22 insertions(+), 9 deletions(-)
/content/gdrive/MyDrive/ece5545


In [4]:
# This extension reloads all imports before running each cell
%load_ext autoreload
%autoreload 2

In [5]:
!ls {PROJECT_ROOT}

1-conv1d_cpu.ipynb  3-conv1d_fpga.ipynb  5-conv2d_dw_gpu.ipynb	README.md  tests
2-conv1d_gpu.ipynb  4-gemm_gpu.ipynb	 leaderboard_id.txt	src


## 2 Install TVM

In [6]:
!pip install numpy==1.24.3

In [7]:
!pip install tlcpack-nightly-cu102 -f https://tlcpack.ai/wheels

Looking in links: https://tlcpack.ai/wheels
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.5/428.5 MB 3.9 MB/s eta 0:00:00


## 3. Implement `make_dwsp_conv2d_gpu_scheduler` function in `src.ops`

In that function, you are required to implemented 2D convolution and use TVM to optimize it.
Please use zero padding and unit stride.
You can assume kernel size to be an odd number.
The padding will equals to kernel size minus ones.
In this case, the output image will preserve the input image dimension.

The `make_dwsp_conv2d_gpu_scheduler` takes following arguments:
1. Batch size $B$;
2. Input channel size $C$;
3. Input image height $H$;
4. Input image width $W$;
5. Output number of channels $O$;
6. Kernel size $K$

You should return both the TVM scheduler and the TVM opterator for
1. Input tensor $x$ with size (B, C, H, W)
2. Input kernel weight $y$ with size (O, 1, K, K)
3. Output $out$ with size (B, O, H, W)

The scheduler should be able to used to build a function with signature $func(x, y, out)$.
Please see the following cells the usage.

In [13]:
import tvm
import torch.nn.functional as F
import numpy as np
import sys
# Adding assignment 3 to the system path
# Make sure this matches your git directory
sys.path.insert(0, PROJECT_ROOT)
from src.ops import make_dwsp_conv2d_gpu_scheduler, make_dwsp_conv2d_gpu_scheduler_baseline

B = 3
C = 4
H = 16
W = 32
K = 7
dtype = 'float32'
a_np = np.random.rand(B, C, H, W).astype(dtype)
w_np = np.random.rand(C, 1, K, K).astype(dtype)

# s, inp, ker, out = make_dwsp_conv2d_gpu_scheduler_baseline(B, C, H, W, K)
s, inp, ker, out = make_dwsp_conv2d_gpu_scheduler(B, C, H, W, K)

func = tvm.build(s, [inp, ker, out], "cuda")

dev = tvm.cuda(0)
a = tvm.nd.array(a_np, dev)
w = tvm.nd.array(w_np, dev)
b = tvm.nd.array(np.zeros((B, C, H, W), dtype), dev)
func(a, w, b)
evaluator = func.time_evaluator(func.entry_name, dev, number=1, repeat =1)

print("Output:", b)
print(f"2DConv TVM: %f ms" % (evaluator(a, w, b).mean * 1e3))

Output: [[[[ 2.8966403  4.230842   5.3547616 ...  6.9960885  6.2801986
     4.6457934]
   [ 3.268038   5.1260204  6.590799  ...  9.227565   7.413626
     5.785445 ]
   [ 3.2522833  6.3138213  6.5362334 ... 10.384426   9.896758
     6.899619 ]
   ...
   [ 5.610786   7.303855   8.852158  ... 10.770857   8.051397
     7.25091  ]
   [ 5.402006   5.5758705  7.1727324 ...  6.5220866  6.72162
     4.350856 ]
   [ 3.624164   4.475112   5.7281146 ...  6.1571326  4.504496
     4.36442  ]]

  [[ 3.7184217  4.6854134  6.5288625 ...  8.5667305  7.442577
     5.471698 ]
   [ 4.6064563  6.3529325  8.378551  ...  8.641472   7.254413
     5.752485 ]
   [ 6.153268   6.819646   8.2764015 ... 10.7836075  8.048927
     6.5557075]
   ...
   [ 6.539511   7.2463584  9.510097  ...  8.700025   6.918482
     5.107772 ]
   [ 5.339259   6.566483   7.073202  ...  7.145812   5.4379554
     4.7617106]
   [ 4.0879593  4.8867254  6.1673512 ...  4.8844113  4.5665994
     3.267554 ]]

  [[ 4.4871416  4.9309015  6.18916  

In [14]:
print(tvm.lower(s, [inp, ker, out], simple_mode=True))

# from tvm.script import ir as I
# from tvm.script import tir as T

@I.ir_module
class Module:
    @T.prim_func
    def main(inp: T.Buffer((3, 4, 16, 32), "float32"), ker: T.Buffer((4, 1, 7, 7), "float32"), out: T.Buffer((3, 4, 16, 32), "float32")):
        T.func_attr({"from_legacy_te_schedule": T.bool(True), "tir.noalias": T.bool(True)})
        inp_padded = T.allocate([10032], "float32", "global")
        inp_padded_1 = T.Buffer((10032,), data=inp_padded)
        for batch, channel in T.grid(3, 4):
            blockIdx_z = T.launch_thread("blockIdx.z", 6)
            blockIdx_y = T.launch_thread("blockIdx.y", 10)
            threadIdx_z = T.launch_thread("threadIdx.z", 4)
            threadIdx_y = T.launch_thread("threadIdx.y", 4)
            if T.likely(blockIdx_z * 2 + threadIdx_z // 2 < 11):
                if T.likely(blockIdx_y * 2 + threadIdx_y // 2 < 19):
                    inp_1 = T.Buffer((6144,), data=inp.data)
                    inp_padded_1[batch * 3344 + channel * 836

In [15]:
%cd {PROJECT_ROOT}
!python -m pytest tests/test_dwsp_2dconv_gpu.py

/content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
======================================= test session starts ========================================
platform linux -- Python 3.11.11, pytest-8.3.5, pluggy-1.5.0
rootdir: /content/gdrive/MyDrive/ece5545/a3-AaronGoldblatt
plugins: anyio-4.9.0, typeguard-4.4.2, langsmith-0.3.23
collected 1357 items                                                                               

tests/test_dwsp_2dconv_gpu.py .............................................................. [  4%]
............................................................................................ [ 11%]
............................................................................................ [ 18%]
............................................................................................ [ 24%]
............................................................................................ [ 31%]
........................................................................